In [6]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt

# Check if GPU is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


In [7]:
# Load preprocessed and normalized dataset
df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/MAI391_Project/hcm_weather_preprocessed.csv')
feature_cols = ['Temp', 'Feels', 'Wind', 'Gust', 'Rain', 'Humidity', 'Cloud', 'Pressure', 'Vis']

data = df[feature_cols].values
target_idx = feature_cols.index('Humidity')

# --- THE FIX: NEUTRALIZE NaNs ---
# Since the data is Z-score normalized, the mean is 0.
# Filling NaNs with 0 safely replaces missing data with the statistical average.
if np.isnan(data).any():
    print("Warning: NaNs detected in input data! Imputing with 0.0...")
    data = np.nan_to_num(data, nan=0.0)

# Hyperparameters
window_size = 7 # 7-day look-back window
X_seq, y_seq = [], []

# Create sequences
for i in range(len(data) - window_size):
    X_seq.append(data[i : i + window_size])
    y_seq.append(data[i + window_size, target_idx])

X_seq = np.array(X_seq)
y_seq = np.array(y_seq)

# Chronological Split (80/20)
split_idx = int(len(X_seq) * 0.8)
X_train, X_test = X_seq[:split_idx], X_seq[split_idx:]
y_train, y_test = y_seq[:split_idx], y_seq[split_idx:]

# Convert to PyTorch Tensors
train_dataset = TensorDataset(torch.tensor(X_train, dtype=torch.float32), torch.tensor(y_train, dtype=torch.float32))
test_dataset = TensorDataset(torch.tensor(X_test, dtype=torch.float32), torch.tensor(y_test, dtype=torch.float32))

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=False)
print(f"Data ready. Training sequences: {len(X_train)}")

Data ready. Training sequences: 3491


In [8]:
class TemporalAttention(nn.Module):
    def __init__(self, hidden_dim):
        super(TemporalAttention, self).__init__()
        self.attention = nn.Linear(hidden_dim, 1)

    def forward(self, lstm_output):
        # lstm_output shape: (batch_size, seq_len, hidden_dim)
        attn_weights = torch.softmax(self.attention(lstm_output), dim=1)
        # Context vector shape: (batch_size, hidden_dim)
        context = torch.sum(attn_weights * lstm_output, dim=1)
        return context, attn_weights

class PITAN(nn.Module):
    def __init__(self, input_dim, hidden_dim, cnn_filters=64):
        super(PITAN, self).__init__()

        # 1D-CNN Feature Extractor
        self.cnn = nn.Conv1d(in_channels=input_dim, out_channels=cnn_filters, kernel_size=3, padding=1)
        self.relu = nn.ReLU()

        # LSTM Encoder
        self.lstm = nn.LSTM(input_size=cnn_filters, hidden_size=hidden_dim, batch_first=True)

        # Temporal Attention Mechanism
        self.attention = TemporalAttention(hidden_dim)

        # Final Regression Head
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        # x shape: (batch, seq_len, features) -> CNN expects (batch, features, seq_len)
        x = x.permute(0, 2, 1)

        # CNN Feature Extraction
        cnn_out = self.relu(self.cnn(x))

        # Permute back for LSTM: (batch, seq_len, cnn_filters)
        cnn_out = cnn_out.permute(0, 2, 1)

        # LSTM Encoding
        lstm_out, _ = self.lstm(cnn_out)

        # Apply Attention
        context_vector, attn_weights = self.attention(lstm_out)

        # Final Prediction
        prediction = self.fc(context_vector)
        return prediction.squeeze()

In [9]:
class PhysicsGuidedLoss(nn.Module):
    def __init__(self, alpha=0.5, upper_bound_z=3.0, lower_bound_z=-3.0):
        super(PhysicsGuidedLoss, self).__init__()
        self.mse = nn.MSELoss()
        self.alpha = alpha
        # Assuming Z-score normalized data, values mostly fall between -3 and 3
        self.upper_bound = upper_bound_z
        self.lower_bound = lower_bound_z

    def forward(self, predictions, targets):
        # 1. Calculate Standard MSE Loss
        loss_mse = self.mse(predictions, targets)

        # 2. Calculate Physics Penalty
        # Penalize predictions that go above upper bound or below lower bound
        penalty_upper = torch.relu(predictions - self.upper_bound)
        penalty_lower = torch.relu(self.lower_bound - predictions)

        # Mean of all constraint violations in the batch
        loss_physics = torch.mean(penalty_upper**2 + penalty_lower**2)

        # 3. Total Hybrid Loss
        loss_total = loss_mse + (self.alpha * loss_physics)

        return loss_total, loss_mse, loss_physics

In [10]:
# Initialize model, loss, and optimizer
input_features = len(feature_cols)
hidden_units = 50

model = PITAN(input_dim=input_features, hidden_dim=hidden_units).to(device)
criterion = PhysicsGuidedLoss(alpha=0.5)
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training Loop
epochs = 20
model.train()

print("Starting Prototype Training...")
for epoch in range(epochs):
    epoch_loss = 0
    epoch_mse = 0
    epoch_phys = 0

    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)

        optimizer.zero_grad()
        predictions = model(batch_X)

        # Compute hybrid loss
        loss_total, loss_mse, loss_physics = criterion(predictions, batch_y)

        loss_total.backward()

        # --- THE FIX: GRADIENT CLIPPING ---
        # Prevents exploding gradients in the LSTM
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()

        epoch_loss += loss_total.item()
        epoch_mse += loss_mse.item()
        epoch_phys += loss_physics.item()

    if (epoch + 1) % 5 == 0:
        print(f"Epoch [{epoch+1}/{epochs}] | Total Loss: {epoch_loss/len(train_loader):.4f} "
              f"| MSE: {epoch_mse/len(train_loader):.4f} | Physics Penalty: {epoch_phys/len(train_loader):.6f}")

print("\nPITAN Prototype training complete.")

Starting Prototype Training...
Epoch [5/20] | Total Loss: 0.1509 | MSE: 0.1509 | Physics Penalty: 0.000000
Epoch [10/20] | Total Loss: 0.1167 | MSE: 0.1167 | Physics Penalty: 0.000000
Epoch [15/20] | Total Loss: 0.1070 | MSE: 0.1070 | Physics Penalty: 0.000000
Epoch [20/20] | Total Loss: 0.0979 | MSE: 0.0979 | Physics Penalty: 0.000000

PITAN Prototype training complete.
